In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

In [57]:
df = pd.read_csv('prices_all.csv')

In [58]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['Ticker', 'date']).reset_index(drop=True)

In [59]:
print(df.head())

        date Ticker      Open      High       Low     Close        Volume
0 2008-01-02   AAPL  5.976315  6.006006  5.774775  5.843454  1.079179e+09
1 2008-01-03   AAPL  5.860551  5.919933  5.778975  5.846155  8.420664e+08
2 2008-01-04   AAPL  5.741786  5.788272  5.365099  5.399888  1.455832e+09
3 2008-01-07   AAPL  5.435876  5.506355  5.105375  5.327609  2.072193e+09
4 2008-01-08   AAPL  5.402587  5.472167  5.122471  5.135967  1.523816e+09


### Поделю на train и test

In [60]:
train_list, test_list = [], []

for ticker, group in df.groupby('Ticker'):
    if len(group) < 300:
        continue
    
    split_date = group['date'].quantile(0.8)
    train_list.append(group[group['date'] < split_date])
    test_list.append(group[group['date'] >= split_date])

train = pd.concat(train_list, ignore_index=True)
test = pd.concat(test_list, ignore_index=True)

### Добавлю target (доходность)

In [61]:
train['target'] = np.log(train['Close'].shift(-1) / train['Close'])
test['target'] = np.log(test['Close'].shift(-1) / test['Close'])

In [62]:
train = train.dropna().reset_index(drop=True)
test = test.dropna().reset_index(drop=True)

### Добавлю признак: скользящие средние от логарифма цены на 20, 50, 250 дней

In [63]:
log_close_shift = np.log(train['Close'].shift(1))

for window in [20, 50, 250]:
    train[f'ma_log_{window}'] = log_close_shift.rolling(window).mean()

log_close_shift = np.log(test['Close'].shift(1))

for window in [20, 50, 250]:
    test[f'ma_log_{window}'] = log_close_shift.rolling(window).mean()

### Дальше добавлю доходности за предыдущие дни

In [64]:
for lag in [1, 2, 3, 5]:
    train[f'return_lag_{lag}'] = np.log(train['Close'].shift(lag) / train['Close'].shift(lag + 1))

for lag in [1, 2, 3, 5]:
    test[f'return_lag_{lag}'] = np.log(test['Close'].shift(lag) / test['Close'].shift(lag + 1))

### Дальше добавлю историческую волатильность

In [65]:
return_1d = np.log(train['Close'].shift(1) / train['Close'].shift(2))
train['volatility'] = return_1d.rolling(20).std().shift(1)

return_1d = np.log(test['Close'].shift(1) / test['Close'].shift(2))
test['volatility'] = return_1d.rolling(20).std().shift(1)

### Дальше добавлю дневной диапазон

In [66]:
train['daily_range'] = (train['High'].shift(1) - train['Low'].shift(1)) / train['Close'].shift(1)

test['daily_range'] = (test['High'].shift(1) - test['Low'].shift(1)) / test['Close'].shift(1)

### Дальше добавлю ценовые разницы

In [67]:
train['high_open_diff'] = (train['High'].shift(1) - train['Open'].shift(1)) / train['Open'].shift(1)
train['open_low_diff'] = (train['Open'].shift(1) - train['Low'].shift(1)) / train['Low'].shift(1)

test['high_open_diff'] = (test['High'].shift(1) - test['Open'].shift(1)) / test['Open'].shift(1)
test['open_low_diff'] = (test['Open'].shift(1) - test['Low'].shift(1)) / test['Low'].shift(1)

### Дальше добавлю (вдруг поможет) временные признаки раздельно

In [68]:
train['day_of_week'] = train['date'].dt.dayofweek
train['month'] = train['date'].dt.month

test['day_of_week'] = test['date'].dt.dayofweek
test['month'] = test['date'].dt.month

### Удалю все что с nan

In [69]:
train = train.dropna().reset_index(drop=True)
test = test.dropna().reset_index(drop=True)

### Подготовка к обучению (выделю признаки), маштабирование и сплит

In [70]:
feat =  ['ma_log_20', 'ma_log_50', 'ma_log_250','return_lag_1', 'return_lag_2', 'return_lag_3', 'return_lag_5',
        'volatility', 'daily_range', 'high_open_diff', 'open_low_diff',
        'day_of_week', 'month']

X_train = train[feat]
y_train = train['target']
X_test = test[feat]
y_test = test['target']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Обучение моделей

In [73]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

print(f"R²: {r2_score(y_test, y_pred_lr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.4f}")
print(f"MAE: {np.mean(np.abs(y_test - y_pred_lr)):.4f}")

R²: -0.0005
RMSE: 0.0524
MAE: 0.0147


In [71]:
model = Ridge(alpha=0.01)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print(f"R²: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE: {np.mean(np.abs(y_test - y_pred)):.4f}")

R²: -0.0005
RMSE: 0.0524
MAE: 0.0147


In [75]:
alphas = [0.0001, 0.001, 0.01, 0.1, 1, 10]
lasso_results = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    y_pred_lasso = lasso.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred_lasso)
    lasso_results.append({'alpha': alpha, 'R²': r2})


best_alpha = max(lasso_results, key=lambda x: x['R²'])['alpha']

lasso_best = Lasso(alpha=best_alpha, max_iter=10000)
lasso_best.fit(X_train_scaled, y_train)
y_pred_lasso = lasso_best.predict(X_test_scaled)

print(f"R²: {r2_score(y_test, y_pred_lasso):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lasso)):.4f}")
print(f"MAE: {np.mean(np.abs(y_test - y_pred_lasso)):.4f}")

R²: -0.0000
RMSE: 0.0524
MAE: 0.0145


In [76]:
alphas = [0.0001, 0.001, 0.01, 0.1, 1]
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
en_results = []

for alpha in alphas:
    for l1_ratio in l1_ratios:
        en = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
        en.fit(X_train_scaled, y_train)
        y_pred_en = en.predict(X_test_scaled)
        r2 = r2_score(y_test, y_pred_en)
        en_results.append({'alpha': alpha, 'l1_ratio': l1_ratio, 'R²': r2})

best_en = max(en_results, key=lambda x: x['R²'])

en_best = ElasticNet(alpha=best_en['alpha'], l1_ratio=best_en['l1_ratio'], max_iter=10000)
en_best.fit(X_train_scaled, y_train)
y_pred_en = en_best.predict(X_test_scaled)

print(f"R²: {r2_score(y_test, y_pred_en):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_en)):.4f}")
print(f"MAE: {np.mean(np.abs(y_test - y_pred_en)):.4f}")

R²: 0.0003
RMSE: 0.0524
MAE: 0.0145


### Выводы:
- Модели предсказивают плохо (ниже чем просто среднее)
- ElasticNet показыват лучшую метрику среди других линейных моделей (но все равно плохую)